# BayMorph — Assess an Observed Single-Band Image

This notebook performs the decisions that must precede inference. It
reads a user-supplied FITS image, asks the user to identify the target,
inspects the observation, evaluates operational flux support, asks which
sampler to use, and writes a reviewed BayMorph YAML configuration.

It does **not** run the production fit. Notebook
`02B_Observed_Fit.ipynb` consumes the YAML created here.

Model screening, posterior inference, and Bayesian model comparison are
distinct. The starting model remains an explicit scientific decision.
Evidence-based comparison is possible only with a backend that estimates
evidence; emcee is posterior-only.


In [ ]:
from pathlib import Path

import yaml

from baymorph import ObservationAdvisor


## 1. Select the observation

Set the FITS path and analysis name. Relative paths are resolved from
the BayMorph project root even when Jupyter runs inside `notebooks`.
Absolute FITS paths are also accepted.


In [ ]:
fits_path = Path("data/observed_galaxy.fits")
analysis_name = "observed_galaxy"
config_path = Path("configs") / f"{analysis_name}.yaml"

advisor = ObservationAdvisor.from_project(fits_path)


## 2. Identify the target

Click the target galaxy in the image. The cyan cross can also be placed
with the numerical (X,Y) fields. BayMorph uses zero-based coordinates:
(X) is the NumPy column and (Y) is the NumPy row.

Press **Confirm center** before continuing. A candidate position is never
accepted implicitly.


In [ ]:
target_selector = advisor.select_target(
    title="Observed image — select the target center"
)

target_selector


## 3. Inspect the FITS header and image

The Advisor reports dimensions, finite-pixel coverage, pixel scale,
seeing metadata, background and noise scales, and diagnostics for the
selected source.

Image-derived flux, size, axis ratio, and position angle are diagnostic
scales. They are not silently converted into scientific prior bounds.
Review the segmentation and unavailable or ambiguous metadata before
continuing.


In [ ]:
inspection = advisor.inspect()

print(advisor.summary())


## 4. Declare the model, prior policy, and sampler

`single_sersic` is the simplest baseline starting model; it is not a
claim that the source is intrinsically single-Sérsic.

Enter one sampler key: `nautilus`, `dynesty-static`,
`dynesty-dynamic`, `pocomc`, or `emcee`. The first four return posterior
samples and evidence. emcee returns posterior samples only and cannot
support Bayes-factor model comparison.

The normalized likelihood provides a stable probabilistic definition
regardless of backend. These numerical configurations are
starting points that must be validated for the supplied observation.
In particular, the emcee preset is a diagnostic pilot, not an automatic
convergence guarantee for broad observational priors. It deliberately
starts with one process; benchmark your likelihood before increasing
parallelism. The YAML Dynesty presets omit the runtime-only `rstate`, so
they are not guaranteed to be bitwise reproducible. If you test another
backend or seed, first assign a new `analysis_name` so previous products
and checkpoints cannot be overwritten.


In [ ]:
starting_model = "single_sersic"

# Define this from a predeclared sample- or instrument-level policy.
# Do not copy target-specific calibration bounds automatically.
# Use a consistent physical prior policy across evidence comparisons.
# Different parameterizations need not have identical component bounds.
reviewed_flux_prior = None

# Expected schema after scientific review:
# reviewed_flux_prior = {
#     "type": "log_uniform",
#     "lower": ...,
#     "upper": ...,
# }

sampler_descriptions = {
    "nautilus": "posterior + evidence",
    "dynesty-static": "posterior + evidence",
    "dynesty-dynamic": "posterior + evidence",
    "pocomc": "posterior + evidence",
    "emcee": "posterior only; no evidence",
}

print("Available samplers:")
for key, description in sampler_descriptions.items():
    print(f"  {key:16s} {description}")

sampler_choice = input("Choose one sampler: ").strip().lower()

if sampler_choice not in sampler_descriptions:
    raise ValueError(
        "Sampler must be one of: "
        + ", ".join(sampler_descriptions)
    )

checkpoint_mode = None
emcee_max_steps = None

if sampler_choice == "emcee":
    checkpoint_mode = (
        input("emcee checkpoint mode [fresh/resume, default fresh]: ")
        .strip()
        .lower()
        or "fresh"
    )

    if checkpoint_mode not in {"fresh", "resume"}:
        raise ValueError(
            "emcee checkpoint mode must be 'fresh' or 'resume'."
        )

    default_steps = 1000 if checkpoint_mode == "fresh" else 3000
    requested_steps = input(
        "Total emcee production steps "
        f"[default {default_steps}]: "
    ).strip()
    emcee_max_steps = int(requested_steps or default_steps)

    if emcee_max_steps < 1:
        raise ValueError("emcee max_steps must be positive.")

serial_execution = {
    "processes": 1,
    "progress": True,
}

sampler_configurations = {
    "nautilus": {
        "name": "nautilus",
        "execution": serial_execution,
        "init": {
            "n_live": 300,
            "seed": 42,
        },
        "run": {
            "n_eff": 500,
            "f_live": 0.02,
        },
    },
    "dynesty-static": {
        "name": "dynesty",
        "mode": "static",
        "execution": serial_execution,
        "init": {
            "nlive": 500,
            "sample": "rwalk",
            "bound": "multi",
        },
        "run": {
            "dlogz": 0.05,
        },
    },
    "dynesty-dynamic": {
        "name": "dynesty",
        "mode": "dynamic",
        "execution": serial_execution,
        "init": {
            "sample": "rwalk",
            "bound": "multi",
        },
        "run": {
            "nlive_init": 500,
            "nlive_batch": 250,
            "dlogz_init": 0.05,
        },
    },
    "pocomc": {
        "name": "pocomc",
        "execution": serial_execution,
        "init": {
            "n_effective": 512,
            "n_active": 256,
            "random_state": 42,
        },
        "run": {
            "n_total": 4096,
            "n_evidence": 4096,
        },
    },
    "emcee": {
        "name": "emcee",
        "execution": serial_execution,
        "init": {
            "nwalkers": "auto",
            "seed": 42,
            "checkpoint_path": "auto",
            "checkpoint_mode": checkpoint_mode,
        },
        "run": {
            "burn_in": 300,
            "max_steps": emcee_max_steps,
            "check_interval": 100,
            "min_autocorr_times": 50.0,
            "tau_tolerance": 0.05,
        },
    },
}

analysis_options = {
    "object_name": analysis_name,
    "outputs_root": "runs",
    "psf_type": "empirical",
    "likelihood": {
        "type": "gaussian",
        "include_normalization": True,
    },
    "sampler": sampler_configurations[sampler_choice],
}

print(f"Selected sampler: {sampler_choice}")
if sampler_choice == "emcee":
    print("Model comparison will remain disabled: no evidence.")


## 5. Evaluate operational flux support

This stage prepares the image and empirical PSF, then evaluates the flux
range over which the declared morphology family is detectable without
saturating the detector.

The measured target flux does not define or centre the bounds. The
calibration output is an operational diagnostic only; it is not inserted
automatically into the scientific configuration. Posterior contact with
a declared prior boundary must still be checked after fitting.

On the first pass, leave `reviewed_flux_prior = None`, run this calibration,
and inspect it. Then return to section 4, enter the independent scientific
prior, rerun that cell, and continue directly to section 6. The expensive
calibration need not be repeated.


In [ ]:
flux_calibration = advisor.calibrate_flux_prior(
    prepare=True,
    build_psf=True,
    snr_threshold=5.0,
    required_detectable_fraction=0.95,
    required_unsaturated_fraction=0.95,
    n_draws=1024,
    random_state=42,
    bootstrap_samples=512,
    confidence_level=0.95,
    model_name=starting_model,
    **analysis_options,
)

print(flux_calibration.summary())
print(
    "The calibrated bounds above are operational diagnostics only; "
    "they are not inserted into the fitting configuration."
)


## 6. Review the proposed configuration

Creating the proposal does not write a file. It records the confirmed
target, user-selected model, independently reviewed flux prior, sampler,
likelihood, and warnings.

Evidence-related warnings apply only to evidence-producing backends. If
emcee is selected, the posterior and diagnostics remain meaningful but
`logZ` and Bayes factors are unavailable.


In [ ]:
proposal = None

if reviewed_flux_prior is None:
    raise ValueError(
        "Define reviewed_flux_prior from a predeclared scientific "
        "policy before creating the fitting configuration."
    )

proposal = advisor.propose_config(
    output_path=config_path,
    model_name=starting_model,
    flux_prior=reviewed_flux_prior,
    **analysis_options,
)

if sampler_choice == "emcee":
    print(
        "emcee selected: this proposal defines posterior inference "
        "without Bayesian evidence."
    )

print(proposal.summary())
print("\nComplete proposed YAML:\n")
print(
    yaml.safe_dump(
        proposal.config,
        sort_keys=False,
    )
)


## 7. Write the YAML

Run this cell only after reviewing the complete proposal. The explicit
`accept=True` is required; without it BayMorph refuses to write the
configuration.


In [ ]:
if proposal is None:
    raise ValueError(
        "Create and review a valid configuration proposal before "
        "writing."
    )

config_path = proposal.write(
    accept=True
)

config_path
